# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [5]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [6]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [7]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [8]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

# First make the patent to citing join like I did in SQL

In [11]:
patent_to_citing = patents.join(citations, patents.PATENT == citations.CITING,"inner").select(patents["*"], citations.CITED)

patent_to_citing.show(10)


+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|  CITED|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------+
|3858242| 1975| 5485|   1973|     US|     MI|    NULL|      1|     5|     2|  6|    63|    4|      12|    0.75| 0.1528|     0.0| 11.0833|   16.25|    NULL|    NULL|    NULL|    NULL|1515701|
|3858242| 1975| 5485|   1973|     US|     MI|    NULL|      1|     5|     2|  6|    63|    4|      12|    0.75| 0.1528|     0.0| 11.0833|   16.25|    NULL|    NULL|    NULL|    NULL|3319261|
|3858242| 1975| 5485|   1973|     US|     MI|

## To make the next step easier, rename the patent/postate columns so we know they are attributed to the CITED columns

In [14]:
cited_patents = patents.select(col("PATENT").alias("CITED_PATENT"), 
                               col("POSTATE").alias("CITED_POSTATE"))
cited_patents.show(10)

+------------+-------------+
|CITED_PATENT|CITED_POSTATE|
+------------+-------------+
|     3070801|         NULL|
|     3070802|           TX|
|     3070803|           IL|
|     3070804|           OH|
|     3070805|           CA|
|     3070806|           PA|
|     3070807|           OH|
|     3070808|           IA|
|     3070809|           AZ|
|     3070810|           IL|
+------------+-------------+
only showing top 10 rows



## Now join the citeds to our main table. Check the relevant columns

In [17]:
patents_to_cited = patent_to_citing.join(cited_patents,
                                         patent_to_citing.CITED == cited_patents.CITED_PATENT,
                                         "inner")
patents_to_cited.select("PATENT",
                        "POSTATE",
                        "CITED",
                        "CITED_POSTATE").show()

+-------+-------+-------+-------------+
| PATENT|POSTATE|  CITED|CITED_POSTATE|
+-------+-------+-------+-------------+
|4483021|     MS|3070803|           IL|
|4133055|     NH|3070803|           IL|
|4253313|   NULL|3070803|           IL|
|5054122|   NULL|3070803|           IL|
|5557807|     FL|3070803|           IL|
|4484363|     CA|3070803|           IL|
|4921141|     CA|3070803|           IL|
|5469579|   NULL|3070803|           IL|
|5850636|     CA|3070803|           IL|
|4400830|     FL|3070805|           CA|
|4058119|     WA|3070807|           OH|
|5295932|   NULL|3070807|           OH|
|5514054|   NULL|3070807|           OH|
|3885255|   NULL|3070811|           CA|
|4060860|     UT|3070811|           CA|
|4195370|     WA|3070811|           CA|
|4385407|     CA|3070811|           CA|
|4407027|     CA|3070811|           CA|
|5890240|     CA|3070811|           CA|
|3916457|   NULL|3070811|           CA|
+-------+-------+-------+-------------+
only showing top 20 rows



## Now filter like we did in the SQL. No grouping/counting yet though

In [18]:
costate_citations = patents_to_cited.filter(
    (col("POSTATE") != "") &
    (col("POSTATE") == col("CITED_POSTATE"))
)
costate_citations.select("PATENT",
                         "POSTATE",
                         "CITED",
                         "CITED_POSTATE").show(20)

+-------+-------+-------+-------------+
| PATENT|POSTATE|  CITED|CITED_POSTATE|
+-------+-------+-------+-------------+
|4067198|     AK|3217791|           AK|
|4676695|     AK|3217791|           AK|
|5190098|     AK|3217791|           AK|
|5238053|     AK|3217791|           AK|
|4075779|     AK|3373523|           AK|
|4130086|     AK|3464385|           AK|
|4178878|     AK|3464385|           AK|
|4344414|     AK|3472314|           AK|
|5618134|     AK|3472314|           AK|
|4205718|     AK|3472314|           AK|
|5172587|     AK|3797257|           AK|
|4944413|     AK|3870156|           AK|
|4014293|     AK|3886905|           AK|
|4344414|     AK|3908753|           AK|
|4205718|     AK|3908753|           AK|
|4304440|     AK|3967854|           AK|
|4557498|     AK|4165888|           AK|
|4742798|     AK|4180012|           AK|
|5697730|     AK|4181448|           AK|
|4245930|     AK|4192630|           AK|
+-------+-------+-------+-------------+
only showing top 20 rows



## Now we can count distinct. This should equal the SQL answer

In [19]:
costate_counts = costate_citations.groupBy("PATENT").agg(
    countDistinct("CITED").alias("COCITED_COUNT")
)
costate_counts.orderBy(
    col("COCITED_COUNT").desc()
).show(13)

+-------+-------------+
| PATENT|COCITED_COUNT|
+-------+-------------+
|5959466|          125|
|5983822|          103|
|6008204|          100|
|5952345|           98|
|5998655|           96|
|5958954|           96|
|5936426|           94|
|5925042|           90|
|5978329|           90|
|5951547|           90|
|5980517|           90|
|5739256|           90|
|5913855|           90|
+-------+-------------+
only showing top 13 rows



## Now augment to the original patents like the instructions want. I replaced nulls with 0 too.

In [20]:
augmented_patents = patents.join(
    costate_counts,
    on="PATENT",
    how="left"
)
augmented_patents = augmented_patents.fillna(
    {"COCITED_COUNT": 0}
)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|COCITED_COUNT|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|            0|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    63| NULL|       9|    NULL| 0.3704|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|            0|
|3070805| 1963|

In [22]:
augmented_patents.orderBy(col("COCITED_COUNT").desc()).show(13)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|COCITED_COUNT|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|          125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|          103|
|6008204| 1999|

## Write the results. I also print the number of partitions to make sure it aligns with the number of "parts" in the output.

In [23]:
augmented_patents.write.mode("overwrite").option("header", True).csv("augmented_patents")

In [24]:
augmented_patents.rdd.getNumPartitions()

3